In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from gliner.model import GLiNER

import torch
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup
import os

In [3]:
train_path = "sample_data.json" # from https://github.com/urchade/GLiNER/blob/main/examples/sample_data.json

with open(train_path, "r") as f:
    data = json.load(f)

In [4]:
# available models: https://huggingface.co/urchade

model = GLiNER.from_pretrained("urchade/gliner_small")

/Users/jackboylan/miniconda/envs/glirel/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [5]:
model

GLiNER(
  (token_rep_layer): TokenRepLayer(
    (bert_layer): TransformerWordEmbeddings(
      (model): DebertaV2Model(
        (embeddings): DebertaV2Embeddings(
          (word_embeddings): Embedding(128004, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
          (dropout): StableDropout()
        )
        (encoder): DebertaV2Encoder(
          (layer): ModuleList(
            (0-5): 6 x DebertaV2Layer(
              (attention): DebertaV2Attention(
                (self): DisentangledSelfAttention(
                  (query_proj): Linear(in_features=768, out_features=768, bias=True)
                  (key_proj): Linear(in_features=768, out_features=768, bias=True)
                  (value_proj): Linear(in_features=768, out_features=768, bias=True)
                  (pos_dropout): StableDropout()
                  (dropout): StableDropout()
                )
                (output): DebertaV2SelfOutput(
                  (dense): Linear(in_feature

In [6]:
from types import SimpleNamespace

# Define the hyperparameters in a config variable
config = SimpleNamespace(
    num_steps=1000, # number of training iteration
    train_batch_size=2, 
    eval_every=100, # evaluation/saving steps
    save_directory="logs", # where to save checkpoints
    warmup_ratio=0.1, # warmup steps
    device='cpu',
    lr_encoder=1e-5, # learning rate for the backbone
    lr_others=5e-5, # learning rate for other parameters
    freeze_token_rep=False, # freeze of not the backbone
    
    # Parameters for set_sampling_params
    max_types=25, # maximum number of entity types during training
    shuffle_types=True, # if shuffle or not entity types
    random_drop=True, # randomly drop entity types
    max_neg_type_ratio=1, # ratio of positive/negative types, 1 mean 50%/50%, 2 mean 33%/66%, 3 mean 25%/75% ...
    max_len=384 # maximum sentence length
)

In [11]:
# Set sampling parameters from config
model.set_sampling_params(
    max_types=config.max_types, 
    shuffle_types=config.shuffle_types, 
    random_drop=config.random_drop, 
    max_neg_type_ratio=config.max_neg_type_ratio, 
    max_len=config.max_len
)

train_loader = model.create_dataloader(data, batch_size=5, shuffle=True)
iter_train_loader = iter(train_loader)
x = next(iter_train_loader)

loss = model(x)
loss

> /Users/jackboylan/Desktop/repos/GLiNER/gliner/model.py(166)compute_score_train()
    165         # compute final entity type representation (FFN)
--> 166         entity_type_rep = self.prompt_rep_layer(entity_type_rep)  # (batch_size, len_types, hidden_size)
    167         num_classes = entity_type_rep.shape[1]  # number of entity types

*** AttributeError: 'Tensor' object has no attribute 'shhape'. Did you mean: 'shape'?
torch.Size([5, 684, 2])
tensor([[[ 0,  0],
         [ 0,  1],
         [ 0,  2],
         ...,
         [ 0,  0],
         [ 0,  0],
         [ 0,  0]],

        [[ 0,  0],
         [ 0,  1],
         [ 0,  2],
         ...,
         [ 0,  0],
         [ 0,  0],
         [ 0,  0]],

        [[ 0,  0],
         [ 0,  1],
         [ 0,  2],
         ...,
         [ 0,  0],
         [ 0,  0],
         [ 0,  0]],

        [[ 0,  0],
         [ 0,  1],
         [ 0,  2],
         ...,
         [ 0,  0],
         [ 0,  0],
         [ 0,  0]],

        [[ 0,  0],
        

In [6]:
def train(model, config, train_data, eval_data=None):
    model = model.to(config.device)

    # Set sampling parameters from config
    model.set_sampling_params(
        max_types=config.max_types, 
        shuffle_types=config.shuffle_types, 
        random_drop=config.random_drop, 
        max_neg_type_ratio=config.max_neg_type_ratio, 
        max_len=config.max_len
    )
    
    model.train()

    # Initialize data loaders
    train_loader = model.create_dataloader(train_data, batch_size=config.train_batch_size, shuffle=True)

    # Optimizer
    optimizer = model.get_optimizer(config.lr_encoder, config.lr_others, config.freeze_token_rep)

    pbar = tqdm(range(config.num_steps))

    if config.warmup_ratio < 1:
        num_warmup_steps = int(config.num_steps * config.warmup_ratio)
    else:
        num_warmup_steps = int(config.warmup_ratio)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=config.num_steps
    )

    iter_train_loader = iter(train_loader)

    for step in pbar:
        try:
            x = next(iter_train_loader)
        except StopIteration:
            iter_train_loader = iter(train_loader)
            x = next(iter_train_loader)

        for k, v in x.items():
            if isinstance(v, torch.Tensor):
                x[k] = v.to(config.device)

        loss = model(x)  # Forward pass
            
        # Check if loss is nan
        if torch.isnan(loss):
            continue

        loss.backward()  # Compute gradients
        optimizer.step()  # Update parameters
        scheduler.step()  # Update learning rate schedule
        optimizer.zero_grad()  # Reset gradients

        description = f"step: {step} | epoch: {step // len(train_loader)} | loss: {loss.item():.2f}"
        pbar.set_description(description)

        if (step + 1) % config.eval_every == 0:

            model.eval()
            
            if eval_data is not None:
                results, f1 = model.evaluate(eval_data["samples"], flat_ner=True, threshold=0.5, batch_size=12,
                                     entity_types=eval_data["entity_types"])

                print(f"Step={step}\n{results}")

            if not os.path.exists(config.save_directory):
                os.makedirs(config.save_directory)
                
            model.save_pretrained(f"{config.save_directory}/finetuned_{step}")

            model.train()

In [ ]:
# modify this to your own test data. For now, evaluation only support fix entity types (but can be easily extended)
eval_data = {
    "entity_types": ["Person", 'Event Reservation'],
    "samples": data[:10]
}

train(model, config, data, eval_data)